This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [1]:
#!pip install keras keras-hub --upgrade -q

In [2]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [3]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## Object detection

이 장에서는 다음 내용을 다룹니다.

* **객체 검출 문제 이해**
* **2단계 및 1단계 객체 검출기**
* **간단한 1단계 검출기를 처음부터 학습**시키는 방법
* **사전 학습된 객체 검출기를 사용**하는 방법
---
객체 탐지는 **이미지에서 관심 있는 객체 주변에 상자(경계 상자)를 그리는 작업**입니다(그림 12.1 참조). 이를 통해 이미지에 어떤 객체가 있는지뿐만 아니라 객체가 어디에 있는지까지 알 수 있습니다. 객체 탐지의 가장 일반적인 **응용 분야**는 다음과 같습니다.

* **개수 세기** - 이미지에 특정 객체가 몇 개 있는지 확인합니다.
* **추적** - 동영상의 모든 프레임에서 객체 탐지를 수행하여 장면에서 객체의 움직임을 시간에 따라 추적합니다.
* **자르기** - 관심 있는 객체가 포함된 이미지 영역을 식별하여 잘라내고, 잘라낸 고해상도 이미지 조각을 분류기 또는 광학 문자 인식(OCR) 모델에 전송합니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch12/object-detection.7c5cbfd4.png" width="600"><br>Figure 12.1: Object detectors draw boxes around objects in an image and label them.</p>

객체 인스턴스에 대한 분할 마스크가 있다면, 마스크를 포함하는 가장 작은 박스의 좌표를 이미 계산할 수 있다고 생각할 수도 있습니다. 그렇다면 이미지 분할만 사용하면 되지 않을까요? 객체 탐지 모델은 정말 필요 없을까요?

맞습니다. **분할은 탐지의 상위 개념**입니다. 분할 모델은 탐지 모델이 제공할 수 있는 모든 정보는 물론, 훨씬 **더 많은 정보를 제공**합니다. 이렇게 풍부한 정보는 상당한 **계산 비용을 수반**합니다. 일반적으로 **우수한 객체 탐지 모델은 이미지 분할 모델보다 훨씬 빠르게 실행**됩니다. 또한 데이터 레이블링 비용도 발생합니다. 분할 모델을 학습시키려면 픽셀 단위의 마스크를 수집해야 하는데, 이는 객체 탐지 모델에 필요한 단순한 바운딩 박스보다 훨씬 더 많은 시간이 소요됩니다.

따라서 픽셀 단위 정보가 필요하지 않은 경우, 예를 들어 **이미지에서 객체의 개수만 세는 경우에는 항상 객체 탐지 모델을 사용하는 것이 좋습니다.**

### Single-stage vs. two-stage object detectors

객체 탐지 아키텍처는 크게 두 가지 범주로 나뉩니다.

* 첫째, **영역 후보를 먼저 추출하는 2단계 탐지기**(영역 기반 합성곱 신경망(R-CNN) 모델)
* 두 번째, RetinaNet이나 You Only Look Once (YOLO) 계열 모델과 같은 **단일 단계 탐지기**
  
작동 방식은 다음과 같습니다.

#### Two-stage R-CNN detectors

영역 기반 컨볼루션 신경망(R-CNN) 모델은 두 단계로 구성됩니다. 

* 첫 번째 단계에서는 **이미지를 입력받아 객체처럼 보이는 영역 주변에 수천 개의 부분적으로 겹치는 경계 상자를 생성**합니다. 이러한 상자를 **영역 제안(region proposal**)이라고 합니다. 이 단계는 그다지 정교하지 않기 때문에, 제안된 영역에 실제로 객체가 포함되어 있는지, 있다면 어떤 객체인지 확실히 알 수 없습니다.

두 번째 단계에서는 **각 영역 제안을 분석하여 미리 정해진 여러 클래스로 분류하는 컨볼루션 신경망**이 역할을 합니다. 이는 9장에서 살펴본 모델들과 유사합니다(그림 12.2 참조). **모든 클래스에 대해 낮은 점수를 받은 영역 제안은 제거**됩니다. 그러면 각 클래스에 대해 높은 클래스 존재 점수를 가진 훨씬 더 적은 수의 상자만 남게 됩니다. 마지막으로, 각 객체 주변의 경계 상자를 더욱 정제하여 중복을 제거하고 최대한 정확하게 만듭니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch12/r-cnn-pipeline.8fe83666.png" width="600"><br> Figure 12.2: An R-CNN first extracts region proposals and then classifies the proposals with a ConvNet (a CNN).</p>

**초기 R-CNN 버전**에서 첫 번째 단계는 공간적 일관성에 대한 정의를 사용하여 객체와 유사한 영역을 식별하는 **선택적 탐색(Selective Search)이라는 휴리스틱 모델**이었습니다. 휴리스틱은 머신 러닝에서 자주 사용되는 용어로, 간단히 말해 "누군가가 임의로 만든 하드코딩된 규칙들의 묶음"을 의미합니다. 이는 일반적으로 학습된 모델(규칙이 자동으로 도출되는 모델)이나 이론 기반 모델과 대비되는 개념으로 사용됩니다. **Faster-R-CNN과 같은 후기 버전의 R-CNN**에서는 박스 생성 단계가 **영역 제안 네트워크(Region Proposal Network)라는 딥 러닝 모델로 변경**되었습니다.

**R-CNN의 2단계 접근 방식**은 실제로 매우 효과적이지만, 처리해야 하는 이미지 하나당 수천 개의 패치를 분류해야 하므로 **계산 비용이 상당히 높습니다**. 따라서 **대부분의 실시간 애플리케이션이나 임베디드 시스템에는 적합하지 않습니다**. 제 생각에는 실제 응용 분야에서는 R-CNN처럼 계산 비용이 많이 드는 객체 탐지 시스템이 일반적으로 필요하지 않습니다. **강력한 GPU를 사용하여 서버 측 추론을 수행하는 경우라면 이전 장에서 살펴본 Segment Anything 모델과 같은 분할 모델을 사용하는 것이 더 효율적**일 가능성이 높기 때문입니다. 반대로 **리소스가 제한적이라면 계산 효율이 더 높은 단일 단계 탐지기**와 같은 객체 탐지 아키텍처를 사용하는 것이 좋습니다.

#### Single-stage detectors

2015년경부터 연구자들과 실무자들은 **단일 딥러닝 모델을 사용하여 바운딩 박스 좌표와 레이블을 동시에 예측하는 단일 단계 검출기(single-stage detector) 아키텍처**를 실험하기 시작했습니다. 단일 단계 검출기의 주요 계열로는 RetinaNet, SSD(Single Shot MultiBox Detectors), 그리고 YOLO(You Only Look Once) 계열이 있습니다. 네, 유명한 밈과 같은 이름입니다. 의도적으로 그렇게 지었습니다.

단일 단계 검출기, 특히 최근의 **YOLO** 버전은 2단계 검출기에 비해 **속도가 훨씬 빠르고 효율성이 뛰어나지만, 정확도 면에서는 약간의 손실이 있을 수** 있습니다. 현재 YOLO는 특히 **실시간 애플리케이션 분야에서 가장 인기 있는 객체 검출 모델**이라고 할 수 있습니다. 매년 새로운 버전이 출시되는데, 흥미롭게도 각 버전은 서로 다른 기관에서 개발하는 경향이 있습니다.

다음 섹션에서는 간소화된 YOLO 모델을 처음부터 구축해 보겠습니다.

### Training a YOLO model from scratch

전반적으로 객체 검출기를 구축하는 것은 다소 복잡한 작업일 수 있습니다. 이론적으로 복잡한 부분이 있는 것은 아니지만, 바운딩 박스와 예측 출력을 처리하는 데 많은 코드가 필요하기 때문입니다. 이해를 돕기 위해 2015년에 출시된 최초의 YOLO 모델을 재현해 보겠습니다. 현재까지 YOLO 모델은 12가지 버전이 있지만, 초기 버전이 다루기 비교적 간단합니다.

#### Downloading the COCO dataset

모델을 만들기 전에 학습에 사용할 데이터가 필요합니다. **COCO 데이터셋[1](Common Objects in Context의 약자)은 가장 잘 알려져 있고 널리 사용되는 객체 탐지 데이터셋 중 하나**입니다. 이 데이터셋은 다양한 출처의 실제 사진과 사람이 직접 작성한 주석으로 구성됩니다. **주석에는 객체 레이블, 바운딩 박스 주석, 그리고 전체 분할 마스크가 포함**됩니다. 여기서는 분할 마스크는 무시하고 **바운딩 박스만 사용**하겠습니다.

2017년 버전의 COCO 데이터셋을 다운로드해 보겠습니다. 오늘날 기준으로 보면 큰 데이터셋은 아니지만, **18GB**에 달하는 이 데이터셋은 이 책에서 사용할 데이터셋 중 가장 큰 규모입니다. 코드를 실행하면서 읽고 있다면, 잠시 숨을 고르는 것도 좋겠습니다.

In [ ]:
import keras
import keras_hub

images_path = keras.utils.get_file(
    "coco",
    "http://images.cocodataset.org/zips/train2017.zip",
    extract=True,
)
annotations_path = keras.utils.get_file(
    "annotations",
    "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
    extract=True,
)

 4045029376/19336861798 ━━━━━━━━━━━━━━━━━━━━ 12:14:58 3us/step

이 데이터를 사용하기 전에 몇 가지 입력 데이터 전처리 작업이 필요합니다. **첫 번째 다운로드 파일에는 레이블이 지정되지 않은 COCO 이미지 파일**이 포함되어 있습니다. **두 번째 다운로드 파일에는 모든 이미지 메타데이터가 JSON 파일 형태로 제공**됩니다. COCO는 각 이미지 파일에 ID를 부여하고, 각 바운딩 박스는 이러한 ID 중 하나와 연결됩니다. 이제 모든 바운딩 박스 데이터와 이미지 데이터를 통합해야 합니다.

**각 바운딩 박스는 이미지의 왼쪽 상단 모서리를 기준으로 x, y, 너비, 높이 픽셀 좌표**를 가지고 있습니다. 데이터를 불러올 때 모든 바운딩 박스 좌표를 [0, 1] 범위의 정사각형으로 변환할 수 있습니다. 이렇게 하면 각 이미지의 크기를 일일이 확인하지 않고도 바운딩 박스를 쉽게 조작할 수 있습니다.

In [ ]:
import json

with open(f"{annotations_path}/annotations/instances_train2017.json", "r") as f:
    annotations = json.load(f)

images = {image["id"]: image for image in annotations["images"]}

def scale_box(box, width, height):
    scale = 1.0 / max(width, height)
    x, y, w, h = [v * scale for v in box]
    x += (height - width) * scale / 2 if height > width else 0
    y += (width - height) * scale / 2 if width > height else 0
    return [x, y, w, h]

metadata = {}
for annotation in annotations["annotations"]:
    id = annotation["image_id"]
    if id not in metadata:
        metadata[id] = {"boxes": [], "labels": []}
    image = images[id]
    box = scale_box(annotation["bbox"], image["width"], image["height"])
    metadata[id]["boxes"].append(box)
    metadata[id]["labels"].append(annotation["category_id"])
    metadata[id]["path"] = images_path + "/train2017/" + image["file_name"]
metadata = list(metadata.values())

방금 불러온 데이터를 살펴보겠습니다.

In [ ]:
len(metadata)

In [ ]:
min([len(x["boxes"]) for x in metadata])

In [ ]:
max([len(x["boxes"]) for x in metadata])

In [ ]:
max(max(x["labels"]) for x in metadata) + 1

In [ ]:
metadata[435]

In [ ]:
[keras_hub.utils.coco_id_to_name(x) for x in metadata[435]["labels"]]

우리는 **117,266개의 이미지**를 가지고 있습니다. 각 이미지에는 **최대 63개의 객체**가 있으며, **각 객체에는 경계 상자가 연결**되어 있습니다. 객체에 사용할 수 있는 **레이블**은 COCO 데이터셋 제작자가 선택한 **91가지**뿐입니다.

KerasHub 유틸리티인 `keras_hub.utils.coco_id_to_name(id)`를 사용하여 이러한 정수 레이블을 사람이 읽을 수 있는 이름으로 변환할 수 있습니다. 이는 8장에서 ImageNet 예측값을 텍스트 레이블로 변환하는 데 사용했던 유틸리티와 유사합니다.

좀 더 구체적으로 이해하기 위해 **예시 이미지를 시각화**해 보겠습니다. Matplotlib을 사용하여 **이미지를 그리는 함수**와 **이 이미지 위에 레이블이 지정된 경계 상자를 그리는 함수**를 정의할 수 있습니다. 이 두 함수는 이 장 전체에서 필요합니다. HSV 색 공간을 활용하면 새로운 레이블이 나타날 때마다 새로운 색상을 생성하는 간단한 방법을 사용할 수 있습니다. 색상의 채도와 밝기는 고정하고 색조만 변경하면 이미지에서 뚜렷하게 구분되는 밝은 색상을 생성할 수 있습니다.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
from matplotlib.patches import Rectangle

color_map = {0: "gray"}

def label_to_color(label):
    if label not in color_map:
        h, s, v = (len(color_map) * 0.618) % 1, 0.5, 0.9
        color_map[label] = hsv_to_rgb((h, s, v))
    return color_map[label]

def draw_box(ax, box, text, color):
    x, y, w, h = box
    ax.add_patch(Rectangle((x, y), w, h, lw=2, ec=color, fc="none"))
    textbox = dict(fc=color, pad=1, ec="none")
    ax.text(x, y, text, c="white", size=10, va="bottom", bbox=textbox)

def draw_image(ax, image):
    ax.set(xlim=(0, 1), ylim=(1, 0), xticks=[], yticks=[], aspect="equal")
    image = plt.imread(image)
    height, width = image.shape[:2]
    hpad = (1 - height / width) / 2 if width > height else 0
    wpad = (1 - width / height) / 2 if height > width else 0
    extent = [wpad, 1 - wpad, 1 - hpad, hpad]
    ax.imshow(image, extent=extent)

이제 새로운 시각화를 사용하여 앞서 살펴본 샘플 이미지[2]를 살펴보겠습니다(그림 12.3 참조).

In [ ]:
sample = metadata[435]
ig, ax = plt.subplots(dpi=300)
draw_image(ax, sample["path"])
for box, label in zip(sample["boxes"], sample["labels"]):
    label_name = keras_hub.utils.coco_id_to_name(label)
    draw_box(ax, box, label_name, label_to_color(label))
plt.show()

18기가바이트에 달하는 입력 데이터를 모두 학습시키는 것도 재미있겠지만, 이 책의 예제들은 사양이 낮은 하드웨어에서도 쉽게 실행할 수 있도록 구성하고자 합니다. 만약 **네 개 이하의 박스가 있는 이미지로만 학습 범위를 제한**한다면, 학습 문제를 단순화하고 데이터 크기를 절반으로 줄일 수 있습니다. 이제 데이터를 섞고, 객체 유형별로 분류된 이미지를 학습에 활용해 보겠습니다. 이렇게 분류된 이미지는 학습에 매우 불리합니다.

In [ ]:
import random

metadata = list(filter(lambda x: len(x["boxes"]) <= 4, metadata))
random.shuffle(metadata)

데이터 로딩은 여기까지입니다! 이제 YOLO 모델 생성을 시작해 봅시다.

#### Creating a YOLO model

앞서 언급했듯이 YOLO 모델은 단일 단계 검출기입니다. 장면에서 모든 후보 객체를 먼저 식별한 다음 객체 영역을 분류하는 기존 방식과는 달리, YOLO는 **바운딩 박스와 객체 레이블을 한 번에 생성**합니다.

저희 모델은 **이미지를 그리드로 나누고 각 그리드 위치에서 바운딩 박스와 클래스 레이블, 두 가지 출력을 예측**합니다. Redmon et al. [3]의 원 논문에서는 그리드 위치당 여러 개의 바운딩 박스를 예측했지만, 저희는 **단순화를 위해 각 그리드 사각형에 하나의 바운딩 박스만 예측**합니다.

대부분의 이미지에서는 객체가 그리드 전체에 고르게 분포되어 있지 않으므로, 이를 고려하여 모델은 그림 12.4와 같이 **각 바운딩 박스와 함께 신뢰도 점수를 출력**합니다. 객체가 감지된 위치에서는 신뢰도 점수가 높게, 객체가 없는 위치에서는 0으로 표시되도록 설계했습니다. 대부분의 그리드 위치에는 객체가 없으므로 0에 가까운 신뢰도 점수가 나타날 것입니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch12/yolo-diagram.b7347ca6.png" width="600"><br> Figure 12.4: YOLO outputs as visualized in the first YOLO paper</p>

컴퓨터 비전 분야의 많은 모델처럼 YOLO 모델은 입력 이미지에서 흥미로운 고수준 특징을 추출하기 위해 컨볼루션 신경망(ConvNet) 백본을 사용합니다. 이는 8장에서 처음 다룬 개념입니다. 논문에서 저자들은 자체 백본 모델을 만들고 ImageNet 데이터셋으로 분류를 위해 사전 학습시켰습니다. 하지만 **우리는 직접 백본을 만드는 대신 KerasHub를 사용하여 사전 학습된 백본을 불러올 수 있습니다.**

이 책에서 지금까지 사용해 온 Xception 백본 대신, 9장에서 처음 언급했던 **ResNet 모델군을 사용**하겠습니다. ResNet은 Xception과 구조가 매우 유사하지만, 풀링 레이어 대신 스트라이드를 사용하여 이미지를 다운샘플링합니다. 11장에서 설명했듯이, 스트라이드 컨볼루션은 입력 이미지의 공간적 위치를 중요하게 고려할 때 더 효과적입니다.

이제 **사전 학습된 모델**과 **이미지 크기 조정을 위한 전처리 파일**을 불러오겠습니다. 이미지 크기는 448 × 448로 조정합니다. 객체 탐지 작업에서 이미지 입력 크기는 매우 중요합니다.

In [ ]:
image_size = 448

backbone = keras_hub.models.Backbone.from_preset(
    "resnet_50_imagenet",
)
preprocessor = keras_hub.layers.ImageConverter.from_preset(
    "resnet_50_imagenet",
    image_size=(image_size, image_size),
)

다음으로, **바운딩 박스와 클래스 예측을 출력하는 새로운 레이어를 추가**하여 백본을 객체 탐지 모델로 만들 수 있습니다. YOLO 논문에서 제안된 구성은 매우 간단합니다. **컨볼루션 신경망(ConvNet) 백본의 출력을 받아 활성화 함수가 중간에 있는 두 개의 완전 연결 레이어를 통과**시킵니다. 그런 다음 **출력을 분할**합니다. **처음 다섯 개의 숫자는 바운딩 박스 예측에 사용**됩니다(**네 개는 박스 크기, 하나는 박스 신뢰도**). 나머지는 그림 12.4에 표시된 클래스 확률 맵에 사용됩니다. 이 맵은 91개의 모든 가능한 레이블에 대한 각 그리드 위치에 대한 분류 예측을 나타냅니다.

이를 글로 표현해 보겠습니다.

In [ ]:
from keras import layers

grid_size = 6
num_labels = 91

inputs = keras.Input(shape=(image_size, image_size, 3))
x = backbone(inputs)
x = layers.Conv2D(512, (3, 3), strides=(2, 2))(x)
x = keras.layers.Flatten()(x)
x = layers.Dense(2048, activation="relu", kernel_initializer="glorot_normal")(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(grid_size * grid_size * (num_labels + 5))(x)
x = layers.Reshape((grid_size, grid_size, num_labels + 5))(x)
box_predictions = x[..., :5]
class_predictions = layers.Activation("softmax")(x[..., 5:])
outputs = {"box": box_predictions, "class": class_predictions}
model = keras.Model(inputs, outputs)

모델 요약을 살펴보면 모델을 더 잘 이해할 수 있습니다.

In [ ]:
model.summary()

**백본 출력은 (batch_size, 14, 14, 2048)** 형태입니다. 즉, 이미지당 401,408개의 float형 출력값이 생성되는데, 이는 완전 연결 레이어에 입력하기에는 너무 많습니다. **스트라이드 컨볼루션 레이어를 사용하여 특징 맵의 크기를 (batch_size, 6, 6, 512**)로 줄여 **이미지당 18,432개의 float형 출력값**으로 만듭니다.

다음으로, **두 개의 완전 연결 레이어를 추가**합니다. 전체 특징 맵을 평탄화하고, ReLU 활성화 함수를 사용하는 완전 연결 레이어를 통과시킨 후, **마지막으로 정확한 개수의 예측값(바운딩 박스와 신뢰도 5개, 각 그리드 위치에서의 각 객체 클래스 91개)을 출력하는 완전 연결 레이어**를 통과시킵니다.

마지막으로, 출력을 다시 6×6 그리드 형태로 재구성하고 **바운딩 박스와 클래스 예측값을 분리**합니다. **분류 출력에는 항상 그렇듯이 소프트맥스를 적용**합니다. 바운딩 박스 출력은 좀 더 특별한 처리가 필요하며, 이 부분은 나중에 자세히 다루겠습니다.

잘 진행되고 있습니다! 분류 레이어를 통해 전체 특징 맵을 평탄화하기 때문에 모든 그리드 검출기가 이미지의 전체 특징을 사용할 수 있으며, 지역성 제약이 없다는 점에 유의하십시오. 이는 의도적인 설계로, 큰 객체는 단일 그리드 셀에만 국한되지 않습니다.

#### Readying the COCO data for the YOLO model

저희 모델은 비교적 간단하지만, 예측 그리드에 맞추기 위해 입력값을 전처리해야 합니다. 각 그리드 검출기는 중심이 그리드 박스 안에 있는 모든 박스를 검출하는 역할을 합니다. 저희 **모델은 박스에 대한 다섯 개의 부동 소수점 값(x, y, w, h, 신뢰도)을 출력**합니다.  **x와 y는 그리드 셀 경계를 기준으로 한 객체의 중심 위치(0~1)**를 나타냅니다. **w와 h는 이미지 크기를 기준으로 한 객체의 크기**를 나타냅니다.

w와 h 값은 이미 학습 데이터에 있습니다. 하지만 **x와 y 값을 그리드 값으로 변환**해야 합니다. 이를 위해 두 가지 유틸리티 함수를 정의해 보겠습니다.

In [ ]:
def to_grid(box):
    x, y, w, h = box
    cx, cy = (x + w / 2) * grid_size, (y + h / 2) * grid_size
    ix, iy = int(cx), int(cy)
    return (ix, iy), (cx - ix, cy - iy, w, h)

def from_grid(loc, box):
    (xi, yi), (x, y, w, h) = loc, box
    x = (xi + x) / grid_size - w / 2
    y = (yi + y) / grid_size - h / 2
    return (x, y, w, h)

이제 훈련 데이터를 새로운 그리드 구조에 맞게 재구성해 보겠습니다. 그리드를 포함하는 데이터셋 길이만큼의 배열 두 개를 생성합니다.

* 첫 번째 배열에는 **클래스 확률 맵**을 저장합니다. 올바른 레이블이 지정된 바운딩 박스와 겹치는 모든 그리드 셀에 레이블을 표시합니다. 코드의 간결성을 위해 박스가 겹치는 부분은 고려하지 않겠습니다.

* 두 번째 배열에는 **박스 자체**를 저장합니다. 모든 박스를 그리드로 이동시키고, 해당 박스의 좌표를 올바른 그리드 셀에 레이블로 지정합니다. 레이블 데이터에서 실제 박스의 신뢰도는 항상 1이고, 그 외의 모든 위치에 대한 신뢰도는 0입니다.

In [ ]:
import numpy as np
import math

class_array = np.zeros((len(metadata), grid_size, grid_size))
box_array = np.zeros((len(metadata), grid_size, grid_size, 5))

for index, sample in enumerate(metadata):
    boxes, labels = sample["boxes"], sample["labels"]
    for box, label in zip(boxes, labels):
        (x, y, w, h) = box
        left, right = math.floor(x * grid_size), math.ceil((x + w) * grid_size)
        bottom, top = math.floor(y * grid_size), math.ceil((y + h) * grid_size)
        class_array[index, bottom:top, left:right] = label

for index, sample in enumerate(metadata):
    boxes, labels = sample["boxes"], sample["labels"]
    for box, label in zip(boxes, labels):
        (xi, yi), (grid_box) = to_grid(box)
        box_array[index, yi, xi] = [*grid_box, 1.0]
        class_array[index, yi, xi] = label

박스 그리기 도우미를 사용하여 YOLO 훈련 데이터를 시각화해 보겠습니다(그림 12.5). 첫 번째 입력 이미지[4] 위에 전체 클래스 활성화 맵을 그리고 박스의 레이블과 함께 신뢰도 점수를 추가합니다.

In [ ]:
def draw_prediction(image, boxes, classes, cutoff=None):
    fig, ax = plt.subplots(dpi=300)
    draw_image(ax, image)
    for yi, row in enumerate(classes):
        for xi, label in enumerate(row):
            color = label_to_color(label) if label else "none"
            x, y, w, h = (v / grid_size for v in (xi, yi, 1.0, 1.0))
            r = Rectangle((x, y), w, h, lw=2, ec="black", fc=color, alpha=0.5)
            ax.add_patch(r)
    for yi, row in enumerate(boxes):
        for xi, box in enumerate(row):
            box, confidence = box[:4], box[4]
            if not cutoff or confidence >= cutoff:
                box = from_grid((xi, yi), box)
                label = classes[yi, xi]
                color = label_to_color(label)
                name = keras_hub.utils.coco_id_to_name(label)
                draw_box(ax, box, f"{name} {max(confidence, 0):.2f}", color)
    plt.show()

draw_prediction(metadata[0]["path"], box_array[0], class_array[0], cutoff=1.0)

마지막으로, `tf.data`를 사용하여 이미지 데이터를 불러오겠습니다. 디스크에서 이미지를 불러와 전처리를 적용한 후 배치 처리합니다. 학습 과정을 모니터링하기 위해 검증 세트도 생성해야 합니다.

In [ ]:
import tensorflow as tf

def load_image(path):
    x = tf.io.read_file(path)
    x = tf.image.decode_jpeg(x, channels=3)
    return preprocessor(x)

images = tf.data.Dataset.from_tensor_slices([x["path"] for x in metadata])
images = images.map(load_image, num_parallel_calls=8)
labels = {"box": box_array, "class": class_array}
labels = tf.data.Dataset.from_tensor_slices(labels)

dataset = tf.data.Dataset.zip(images, labels).batch(16).prefetch(2)
val_dataset, train_dataset = dataset.take(500), dataset.skip(500)

이로써 데이터는 학습 준비가 완료되었습니다.

```
이 학습 예제는 tf.data와 같은 스트리밍 라이브러리가 왜 유용한지 명확하게 보여줍니다. 이 대규모 데이터셋의 모든 이미지를 한 번에 로드하면 시스템 메모리에 과부하가 걸립니다(이미지 텐서는 압축된 JPEG 파일보다 훨씬 크다는 점을 기억하세요). tf.data를 사용하면 이미지 데이터를 배치 단위로 로드하고 완료되면 메모리를 해제할 수 있으며, 특정 시점에 필요한 데이터셋 부분만 로드할 수 있습니다. `prefetch(2)` 호출은 tf.data가 두 배치를 미리 버퍼링하고 준비 상태로 유지하도록 하여, 각 배치 학습을 중단하고 추가 이미지를 로드하고 크기를 조정하는 작업을 방지합니다.
```

#### Training the YOLO model

모델과 훈련 데이터는 준비되었지만, `fit()`을 실행하기 전에 마지막으로 필요한 요소가 하나 더 있습니다. 바로 손실 함수입니다. 우리 모델은 예측된 박스와 예측된 그리드 레이블을 출력합니다. 7장에서 살펴본 것처럼 각 출력에 대해 여러 개의 손실 함수를 정의할 수 있으며, Keras는 훈련 중에 이러한 손실 함수들을 단순히 합산합니다. 분류 손실은 평소처럼 `sparse_categorical_crossentropy`를 사용하여 처리할 수 있습니다.

하지만 박스 손실은 특별한 고려가 필요합니다. YOLO 개발자들이 제안한 기본 손실 함수는 비교적 간단합니다. **목표 박스 매개변수와 예측된 박스 매개변수 간의 차이에 대한 제곱 오차의 합**을 사용합니다. 우리는 **레이블이 지정된 데이터에서 실제 박스가 있는 그리드 셀에 대해서만 이 오차를 계산**할 것입니다.

손실 함수에서 까다로운 부분은 박스 신뢰도 출력입니다. 개발자들은 신뢰도 출력이 단순히 객체의 존재 여부뿐만 아니라 예측된 박스의 정확도까지 반영하기를 원했습니다. **박스 예측의 정확도를 부드럽게 추정**하기 위해, 개발자들은 지난 장에서 살펴본 **IoU(Intersection over Union) 지표**를 사용할 것을 제안했습니다. 그리드 셀이 비어 있으면 해당 위치의 예측 신뢰도는 0이어야 합니다. 하지만 그리드 셀에 객체가 있는 경우, 현재 바운딩 박스 예측과 실제 바운딩 박스 사이의 IoU 점수를 목표 신뢰도 값으로 사용할 수 있습니다. 이렇게 하면 모델이 바운딩 박스 위치 예측에 더 능숙해질수록 IoU 점수와 학습된 신뢰도 값이 증가합니다.

이를 위해서는 사용자 정의 손실 함수가 필요합니다. 먼저 **목표 바운딩 박스와 예측 바운딩 박스에 대한 IoU 점수를 계산하는 유틸리티 함수를 정의**할 수 있습니다.

In [ ]:
from keras import ops

def unpack(box):
    return box[..., 0], box[..., 1], box[..., 2], box[..., 3]

def intersection(box1, box2):
    cx1, cy1, w1, h1 = unpack(box1)
    cx2, cy2, w2, h2 = unpack(box2)
    left = ops.maximum(cx1 - w1 / 2, cx2 - w2 / 2)
    bottom = ops.maximum(cy1 - h1 / 2, cy2 - h2 / 2)
    right = ops.minimum(cx1 + w1 / 2, cx2 + w2 / 2)
    top = ops.minimum(cy1 + h1 / 2, cy2 + h2 / 2)
    return ops.maximum(0.0, right - left) * ops.maximum(0.0, top - bottom)

def intersection_over_union(box1, box2):
    cx1, cy1, w1, h1 = unpack(box1)
    cx2, cy2, w2, h2 = unpack(box2)
    intersection_area = intersection(box1, box2)
    a1 = ops.maximum(w1, 0.0) * ops.maximum(h1, 0.0)
    a2 = ops.maximum(w2, 0.0) * ops.maximum(h2, 0.0)
    union_area = a1 + a2 - intersection_area
    return ops.divide_no_nan(intersection_area, union_area)

이 유틸리티를 사용하여 사용자 지정 손실 함수를 정의해 보겠습니다. Redmon 외 연구진은 학습 품질을 향상시키기 위해 몇 가지 손실 스케일링 기법을 제안했습니다.

* 먼저 **박스 위치 손실을 5배로 확대**하여 전체 학습에서 더 중요한 요소로 만듭니다.

* 대부분의 그리드 셀이 비어 있으므로, **빈 위치의 신뢰도 손실은 2배로 줄입**니다. 이렇게 하면 신뢰도가 0인 예측값이 손실 함수에 과도하게 반영되는 것을 방지할 수 있습니다.
* 또한 **손실을 계산하기 전에 너비와 높이의 제곱근을 취합**니다. 이는 큰 박스가 작은 박스보다 불균형적으로 더 큰 영향을 미치는 것을 막기 위함입니다. 학습 초기에는 **너비와 높이가 음수로 예측될 수 있으므로 입력값의 부호를 유지하는 제곱근 함수를 사용**하겠습니다.

이를 코드로 작성해 보겠습니다.

In [ ]:
def signed_sqrt(x):
    return ops.sign(x) * ops.sqrt(ops.absolute(x) + keras.config.epsilon())

def box_loss(true, pred):
    xy_true, wh_true, conf_true = true[..., :2], true[..., 2:4], true[..., 4:]
    xy_pred, wh_pred, conf_pred = pred[..., :2], pred[..., 2:4], pred[..., 4:]
    no_object = conf_true == 0.0
    xy_error = ops.square(xy_true - xy_pred)
    wh_error = ops.square(signed_sqrt(wh_true) - signed_sqrt(wh_pred))
    iou = intersection_over_union(true, pred)
    conf_target = ops.where(no_object, 0.0, ops.expand_dims(iou, -1))
    conf_error = ops.square(conf_target - conf_pred)
    error = ops.concatenate(
        (
            ops.where(no_object, 0.0, xy_error * 5.0),
            ops.where(no_object, 0.0, wh_error * 5.0),
            ops.where(no_object, conf_error * 0.5, conf_error),
        ),
        axis=-1,
    )
    return ops.sum(error, axis=(1, 2, 3))

이제 드디어 YOLO 모델 학습을 시작할 준비가 되었습니다. 예제를 간결하게 유지하기 위해 지표 분석은 생략하겠습니다. 실제 환경에서는 다양한 신뢰도 임계값에서의 모델 정확도와 같은 여러 지표가 필요할 것입니다.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(2e-4),
    loss={"box": box_loss, "class": "sparse_categorical_crossentropy"},
)
model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=4,
)

Colab의 무료 GPU 런타임에서 학습하는 데 한 시간 이상이 걸렸고, 모델은 여전히 ​​학습이 부족한 상태입니다(검증 손실이 계속 감소하고 있습니다!). 이제 모델의 출력 결과를 시각화해 보겠습니다(그림 12.6). 우리 모델은 아직 객체 탐지 ​​성능이 그다지 좋지 않으므로 낮은 신뢰도 임계값을 사용하겠습니다.

In [ ]:
x, y = next(iter(val_dataset.rebatch(1)))
preds = model.predict(x)
boxes = preds["box"][0]
classes = np.argmax(preds["class"][0], axis=-1)
path = metadata[0]["path"]
draw_prediction(path, boxes, classes, cutoff=0.1)

모델이 박스 위치와 클래스 레이블을 이해하기 시작했지만 아직 정확도가 높지는 않다는 것을 알 수 있습니다. 이제 모델이 예측한 모든 박스를 시각화해 보겠습니다(그림 12.7). 신뢰도가 0인 박스까지 모두 포함됩니다.

In [ ]:
draw_prediction(path, boxes, classes, cutoff=None)

우리 모델은 장면에서 객체의 위치를 ​​일관되게 찾는 방법을 아직 학습하지 못했기 때문에 매우 낮은 신뢰도 값을 학습합니다. 모델을 더욱 개선하기 위해 다음과 같은 몇 가지 방법을 시도해 볼 수 있습니다.

더 많은 에포크 동안 학습
전체 COCO 데이터셋 사용
데이터 증강(예: 입력 이미지 및 바운딩 박스의 이동 및 회전)
겹치는 바운딩 박스에 대한 클래스 확률 맵 개선
더 큰 출력 그리드를 사용하여 그리드 위치당 여러 개의 바운딩 박스 예측
이러한 모든 방법은 모델 성능에 긍정적인 영향을 미치고 원래의 YOLO 학습 방식에 더 가까워지도록 할 것입니다. 하지만 이 예제는 객체 탐지 ​​학습에 대한 감을 잡기 위한 것일 뿐입니다. 정확한 COCO 탐지 모델을 처음부터 학습하려면 막대한 컴퓨팅 자원과 시간이 소요됩니다. 따라서 더 나은 성능의 탐지 모델을 경험하기 위해 RetinaNet이라는 사전 학습된 객체 탐지 ​​모델을 사용해 보겠습니다.

### Using a pretrained RetinaNet detector

**RetinaNet 역시 단일 단계 객체 검출기**이며 **YOLO 모델과 동일한 기본 원리**로 작동합니다. 저희 모델과 RetinaNet의 가장 큰 개념적 차이점은 RetinaNet이 **하위 ConvNet을 다르게 활용하여 크고 작은 객체를 동시에 더 효과적으로 처리**한다는 점입니다.

YOLO 모델에서는 ConvNet의 최종 출력값을 그대로 사용하여 객체 검출기를 구축했습니다. 이러한 출력 특징은 입력 이미지의 넓은 영역에 매핑되므로 장면에서 작은 객체를 찾는 데는 효과적이지 않습니다.

이러한 **규모 문제를 해결**하는 한 가지 방법은 **ConvNet의 초기 레이어 출력을 직접 사용하는 것**입니다. 이렇게 하면 **입력 이미지의 작은 영역에 매핑되는 고해상도 특징을 추출**할 수 있습니다. 그러나 이러한 초기 레이어의 출력은 의미론적으로 그다지 중요하지 않습니다. 가장자리나 곡선과 같은 단순한 특징에는 매핑될 수 있지만, 전체 객체에 대한 잠재적 표현을 구축하기 시작하는 것은 ConvNet의 후반 레이어에서입니다.

RetinaNet에서 사용하는 해결책은 **특징 피라미드 네트워크**라고 합니다. 컨볼루션 신경망(ConvNet) 기본 모델의 최종 특징은 이전 장에서 살펴본 것처럼 **점진적인 Conv2DTranspose 레이어를 사용하여 업샘플링**됩니다. 하지만 중요한 점은, **업샘플링된 특징 맵과 원래 컨볼루션 신경망의 동일 크기 특징 맵을 합산하는 측면 연결을 추가**했다는 것입니다. 이를 통해 컨볼루션 신경망 끝부분의 의미적으로 흥미로운 저해상도 특징과 컨볼루션 신경망 시작 부분의 고해상도 소규모 특징을 결합할 수 있습니다. 이 아키텍처의 개략도는 그림 12.8에 나와 있습니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch12/feature-pyramid-network.83b6f108.png" width="600"><br> Figure 12.8: A feature pyramid network creates semantically interesting feature maps at different scales.</p>

특징 피라미드 네트워크는 픽셀 크기 측면에서 크고 작은 객체 모두에 대해 효과적인 특징을 구축함으로써 성능을 크게 향상시킬 수 있습니다. **YOLO의 최근 버전도 동일한 설정을 사용**합니다.

이제 **COCO 데이터셋으로 학습된 RetinaNet 모델을 사용**해 보겠습니다. 좀 더 흥미롭게 하기 위해, 모델의 데이터셋에 포함되지 않은 이미지인 점묘화 '라 그랑 자트 섬의 일요일 오후'를 사용해 보겠습니다.

먼저 이미지를 다운로드하고 NumPy 배열로 변환해 보겠습니다.

In [ ]:
url = "https://s3.us-east-1.amazonaws.com/book.keras.io/3e/seurat.jpg"
path = keras.utils.get_file(origin=url)
image = np.array([keras.utils.load_img(path)])

다음으로, 모델을 다운로드하고 예측을 수행해 보겠습니다. 이전 장에서 했던 것처럼 KerasHub의 고급 작업 API를 사용하여 ObjectDetector를 생성하고 전처리까지 포함하여 사용할 수 있습니다.

In [ ]:
detector = keras_hub.models.ObjectDetector.from_preset(
    "retinanet_resnet50_fpn_v2_coco",
    bounding_box_format="rel_xywh",
)
predictions = detector.predict(image)

보시다시피, 바운딩 박스 형식을 지정하기 위해 추가 인수를 전달합니다. 바운딩 박스를 지원하는 대부분의 Keras 모델과 레이어에 대해 이 방법을 사용할 수 있습니다. YOLO 모델에서 사용했던 것과 동일한 형식을 사용하기 위해 "rel_xywh"를 전달하므로, 동일한 박스 그리기 유틸리티를 사용할 수 있습니다. 여기서 rel은 이미지 크기에 대한 상대적인 값(예: [0, 1])을 의미합니다. 방금 수행한 예측 결과를 살펴보겠습니다.

In [ ]:
[(k, v.shape) for k, v in predictions.items()]

In [ ]:
predictions["boxes"][0][0]

이 모델은 바운딩 박스, 신뢰도, 레이블, 그리고 총 탐지 개수라는 ​​네 가지 출력값을 제공합니다. 이는 YOLO 모델과 전반적으로 매우 유사합니다. 이 모델은 각 입력에 대해 총 100개의 객체를 예측할 수 있습니다.

이제 박스 그리기 유틸리티를 사용하여 예측 결과를 표시해 보겠습니다(그림 12.9).

In [ ]:
fig, ax = plt.subplots(dpi=300)
draw_image(ax, path)
num_detections = predictions["num_detections"][0]
for i in range(num_detections):
    box = predictions["boxes"][0][i]
    label = predictions["labels"][0][i]
    label_name = keras_hub.utils.coco_id_to_name(label)
    draw_box(ax, box, label_name, label_to_color(label))
plt.show()

RetinaNet 모델은 이러한 유형의 입력에 대한 학습이 전혀 없었음에도 불구하고 점묘화에 쉽게 일반화할 수 있습니다! 이는 단일 단계 객체 검출기의 장점 중 하나입니다. 그림과 사진은 픽셀 수준에서는 매우 다르지만, 전체적인 구조는 유사합니다. 반면 R-CNN과 같은 이중 단계 검출기는 입력 이미지의 작은 영역들을 개별적으로 분류해야 하는데, 특히 작은 픽셀 영역들이 학습 데이터와 매우 다를 경우 더욱 어렵습니다. 단일 단계 검출기는 전체 입력 이미지의 특징을 활용할 수 있으며, 새로운 테스트 입력에 대해서도 더욱 강건합니다.

이로써 이 책의 컴퓨터 비전 부분을 마무리하게 되었습니다! 우리는 이미지 분류기, 분할기, 그리고 객체 검출기를 처음부터 학습시켰습니다. 딥러닝 시대의 첫 번째 주요 성공 사례인 컨볼루션 신경망(ConvNet)의 작동 방식에 대한 좋은 직관을 얻었습니다. 하지만 이미지에 대한 이야기는 아직 끝나지 않았습니다. 17장에서 이미지 출력을 생성하기 시작할 때 다시 한번 이미지를 접하게 될 것입니다.

### Summary

* 객체 탐지는 경계 상자를 사용하여 이미지 내의 객체를 식별하고 위치를 파악합니다. 기본적으로 이미지 분할의 간소화된 버전이지만 훨씬 효율적으로 실행될 수 있습니다.
* 객체 탐지에는 크게 두 가지 접근 방식이 있습니다.
  - 영역 기반 합성곱 신경망(R-CNN)은 두 단계로 구성된 모델로, 먼저 관심 영역을 제안한 다음 합성곱 신경망(ConvNet)을 사용하여 분류합니다.
  - 단일 단계 탐지기(RetinaNet 및 YOLO 등)는 두 작업을 한 단계에서 수행합니다. 단일 단계 탐지기는 일반적으로 더 빠르고 효율적이므로 실시간 애플리케이션(예: 자율주행 자동차)에 적합합니다.
* YOLO는 학습 중에 가능한 경계 상자와 클래스 확률 맵이라는 두 가지 출력을 동시에 계산합니다.
  - 각 후보 경계 상자는 신뢰도 점수와 쌍을 이루며, 이 점수는 예측된 상자와 실제 상자의 IOU(Intersection over Union) 값을 목표로 학습됩니다.
  - 클래스 확률 맵은 이미지의 서로 다른 영역이 서로 다른 객체에 속하는지 분류합니다.

* RetinaNet은 여러 ConvNet 레이어의 특징을 결합하여 다양한 크기의 특징 맵을 생성하는 특징 피라미드 네트워크(FPN)를 사용하여 이러한 아이디어를 기반으로 구축되었으며, 이를 통해 다양한 크기의 객체를 더욱 정확하게 감지할 수 있습니다.